In [0]:
# Read Single line JSON file

df_single = spark.read.format("json").load("/Volumes/workspace/pyspark/input/order_singleline.json")

df_single.show(truncate=False)

In [0]:
df_single.printSchema()

In [0]:
# Read Multiline JSON file

df_multi = spark.read.format("json").option("multiLine", True).load("/Volumes/workspace/pyspark/input/order_multiline.json")

df_multi.show(truncate=False)

In [0]:
df = spark.read.format("text").load("/Volumes/workspace/pyspark/input/order_singleline.json")
df.show(truncate=False)
df_multi.printSchema()

In [0]:
# With Schema

_schema = "customer_id string, order_id string, contact array<long>"

df_schema = spark.read.format("json").schema(_schema).load("/Volumes/workspace/pyspark/input/order_singleline.json")

df_schema.show(truncate=False)

In [0]:
df_schema.printSchema()

In [0]:
_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

df_schema_new = spark.read.format("json").schema(_schema).load("/Volumes/workspace/pyspark/input/order_singleline.json")

df_schema_new.show(truncate=False)

In [0]:
df_schema_new.printSchema()

In [0]:
# Function from_json to read from a column

_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

from pyspark.sql.functions import from_json

df_expanded = df.withColumn("parsed", from_json(df.value, _schema))

df_expanded.printSchema()

In [0]:
df_expanded.show(truncate=False)

In [0]:
# Function to_json to parse a JSON string
from pyspark.sql.functions import to_json

df_unparsed = df_expanded.withColumn("unparsed", to_json(df_expanded.parsed))

df_unparsed.printSchema()

In [0]:
df_unparsed.select("unparsed").show(truncate=False)

In [0]:
# Get values from Parsed JSON

df_1 = df_expanded.select("parsed.*")

In [0]:
from pyspark.sql.functions import explode

df_2 = df_1.withColumn("expanded_line_items", explode("order_line_items"))

df_2.show()

In [0]:
df_3 = df_2.select("contact", "customer_id", "order_id", "expanded_line_items.*")

df_3.show()

In [0]:
# Explode Array fields
df_final = df_3.withColumn("contact_expanded", explode("contact"))
df_final.printSchema()

In [0]:
df_final.drop("contact").show()